# Maze Solver Training - Modular Version

This notebook uses modular code from `src/` folder:
- `src/model.py`: ResNet + GPT2 model architecture
- `src/tokenizer.py`: Simple tokenizer for maze sequences
- `src/dataset.py`: Dataset and DataLoader utilities
- `src/train_utils.py`: Training and evaluation functions

In [5]:
# ----------------------------------------------------------------------
# GPU / Environment Sanity Check
# Run this cell first before anything else to confirm:
#   1. The runtime has a GPU attached (if not: Runtime > Change runtime type)
#   2. PyTorch can see the GPU (rules out CUDA/driver version mismatches)
#   3. We're in the right working directory (important when running on Colab
#      via Google Drive mount or git clone)
# ----------------------------------------------------------------------

import os
import torch
os.listdir('/content')

# 1. Move into the repo root if we're not already there (Colab git clone lands in /content/)
if not os.path.exists('src'):
    for candidate in ['/content/ee641-final', '/content/drive/MyDrive/ee641-final']:
        if os.path.exists(os.path.join(candidate, 'src')):
            os.chdir(candidate)
            print(f"Working directory set to: {candidate}")
            break
    else:
        print("WARNING: Could not find repo root — make sure you've cloned/mounted the repo")
else:
    print(f"Working directory: {os.getcwd()}")

# 2. Show GPU info — confirms which GPU Colab assigned this session
os.system("nvidia-smi")

# 3. Confirm PyTorch sees the GPU and print device name
if torch.cuda.is_available():
    print(f"\nPyTorch CUDA: OK — {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\nNo GPU detected — go to Runtime > Change runtime type and select a GPU.")


PyTorch CUDA: OK — NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [3]:
import sys
import json
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

# Add src to path
sys.path.append('./src')

# Import our modules
from config import get_config
from model import ResNetGPT2PrefixModel
from tokenizer import SimpleTokenizer
from dataset import MazeDataset, collate_fn
from train_utils import train_model, test_model

print("All modules imported successfully")

ModuleNotFoundError: No module named 'config'

## 1. Setup Tokenizer

In [ ]:
print("=" * 60)
print("TOKENIZER SETUP")
print("=" * 60)

tokenizer = SimpleTokenizer()

print(f"Vocabulary: {tokenizer.vocab}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}")

## 2. Load Data

In [ ]:
print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load JSON data
with open('data/train_sequences.json', 'r') as f:
    train_data = json.load(f)
    train_entries = train_data['entries']  # <-- Access the 'entries' key
    train_metadata = train_data['metadata']

with open('data/test_sequences.json', 'r') as f:
    test_data = json.load(f)
    test_entries = test_data['entries']  # <-- Access the 'entries' key
    test_metadata = test_data['metadata']

print(f"Training set: {len(train_entries)} examples")
print(f"Test set: {len(test_entries)} examples")

# Print metadata
print("\n" + "=" * 60)
print("DATASET METADATA")
print("=" * 60)
print(f"Grid size: {train_metadata['grid_size']}×{train_metadata['grid_size']}")
print(f"Start position: {train_metadata['start_position']}")
print(f"Goal position: {train_metadata['goal_position']}")
print(f"Variations per solution: {train_metadata['variations']}")
print(f"Seed: {train_metadata['seed']}")

# Verify sample
print("\n" + "=" * 60)
print("VERIFICATION - Sample Entry")
print("=" * 60)
sample = train_entries[0]
print(f"Sample ID: {sample['id']}")
print(f"Solution ID: {sample['solution_id']}")
print(f"Variation: {sample['variation']}")
print(f"Image path: {sample['image']}")
print(f"Sample sequence: {sample['sequence']}")
token_ids = tokenizer.encode(sample['sequence'])
print(f"Token IDs: {token_ids}")
print(f"Decoded: '{tokenizer.decode_to_string(token_ids)}'")

## 3. Create Datasets and DataLoaders

In [ ]:
# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = MazeDataset(train_entries, tokenizer, transform)
test_dataset = MazeDataset(test_entries, tokenizer, transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

print(f"✓ Train loader: {len(train_loader)} batches")
print(f"✓ Test loader: {len(test_loader)} batches")

## 4. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Select config based on the grid size in the loaded dataset
grid_size = train_metadata['grid_size']
cfg = get_config(grid_size)

print(f"Grid size: {grid_size}×{grid_size}")
print(f"Model config: {cfg['model_kwargs']}")
print(f"Train config: {cfg['train_kwargs']}")

model = ResNetGPT2PrefixModel(
    vocab_size=len(tokenizer),
    **cfg['model_kwargs']
)
model = model.to(device)

print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Prefix tokens: {model.num_prefix_tokens}")

## 5. Train Model

In [ ]:
print("\n" + "=" * 60)
print("TRAINING PHASE")
print("=" * 60)
print(f"Training on {len(train_entries)} mazes for {cfg['train_kwargs']['epochs']} epochs...")
print("=" * 60)

final_loss = train_model(model, train_loader, device=device, **cfg['train_kwargs'])

print(f"\nTraining completed. Final loss: {final_loss:.6f}")

## 6. Evaluate on Training Set

In [ ]:
print("\n" + "=" * 60)
print("TRAINING SET EVALUATION")
print("=" * 60)
print(f"Evaluating model performance on training set ({len(train_entries)} mazes)...")
print("=" * 60)

train_correct = test_model(model, train_loader, device, tokenizer, num_samples=len(train_entries))

print("=" * 60)
print(f"Training Accuracy: {train_correct}/{len(train_entries)} ({100*train_correct/len(train_entries):.1f}%)")
print("=" * 60)

## 7. Evaluate on Test Set

In [ ]:
print("\n" + "=" * 60)
print("TEST SET EVALUATION (UNSEEN DATA)")
print("=" * 60)
print(f"Evaluating on held-out test set ({len(test_entries)} mazes)...")
print("=" * 60)

test_correct = test_model(model, test_loader, device, tokenizer, num_samples=len(test_entries))

print("=" * 60)
print(f"Test Accuracy: {test_correct}/{len(test_entries)} ({100*test_correct/len(test_entries):.1f}%)")
print("=" * 60)

In [ ]:
# Check if model ever predicts EOS token (ID=2)
model.eval()
predictions_with_eos = 0
total_samples = 0

with torch.no_grad():
    for batch in train_loader:
        images = batch['images'].to(device)
        
        preds = model.generate(
            images, 
            max_length=12,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            bos_token_id=tokenizer.bos_token_id
        )
        
        # Check if any prediction contains EOS (token 2)
        for pred in preds:
            total_samples += 1
            if 2 in pred.cpu().tolist():
                predictions_with_eos += 1
        
        if total_samples >= 100:  # Check first 100 samples
            break

print(f"Predictions with EOS token: {predictions_with_eos}/{total_samples}")
print(f"Percentage: {100*predictions_with_eos/total_samples:.1f}%")

## 8. Final Results & Save Model

In [ ]:
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

train_acc_pct = 100 * train_correct / len(train_entries)
test_acc_pct = 100 * test_correct / len(test_entries)
gen_gap = train_acc_pct - test_acc_pct

print(f"Final Training Loss:    {final_loss:.6f}")
print(f"Training Accuracy:      {train_correct}/{len(train_entries)} ({train_acc_pct:.1f}%)")
print(f"Test Accuracy:          {test_correct}/{len(test_entries)} ({test_acc_pct:.1f}%)")
print(f"Generalization Gap:     {gen_gap:.1f}%")
print("=" * 60)

# Performance assessment
if final_loss < 0.1 and test_acc_pct >= 80:
    print("\n🎉 SUCCESS! Model generalizes well to unseen mazes!")
elif final_loss < 0.2 and test_acc_pct >= 60:
    print("\n✓ Good progress! Model is learning but could improve")
else:
    print("\n⚠️  Model may need more training or hyperparameter tuning")
    if gen_gap > 50:
        print("   → High generalization gap suggests overfitting")
    elif train_acc_pct < 50:
        print("   → Low training accuracy suggests underfitting or insufficient capacity")

# Save model
checkpoint_path = "models/resnet_gpt2_prefix.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
    'hidden_size': model.hidden_size,
    'num_prefix_tokens': model.num_prefix_tokens,
    'final_loss': final_loss,
    'train_accuracy': train_acc_pct,
    'test_accuracy': test_acc_pct,
    'generalization_gap': gen_gap,
}, checkpoint_path)

print(f"\n💾 Model checkpoint saved to {checkpoint_path}")
print("=" * 60)